We will use python to scrape the the first goal of the other team. The reason for this is that it was not possible anymore to use R for webscraping.

In [302]:
# Test cell
import undetected_chromedriver as uc
import time
import random

url = "https://fbref.com/en/matches/74ee6b62/Union-Berlin-Augsburg-September-19-2020-Bundesliga"

def get_raw_html(url=url):
    print("1. Start browser")
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options)
    html_content = "" 
    try:
        print(f"2. Go to url")
        driver.get(url)

        print("3. Loading page (please wait)...")
        time.sleep(random.uniform(8, 10))
        html_content = driver.page_source
        
        if "403 Forbidden" in html_content:
            print("blokked")
        else:
            print("SUCCES")

    except Exception as e:
        print(f"error: {e}")

    finally:
        print("4. Closing browser.")
        driver.quit()
        return html_content

html = get_raw_html()

1. Start browser
2. Go to url
3. Loading page (please wait)...
error: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=142.0.7444.176); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x1124093
	0x11240d4
	0xf2e71d
	0xf1d910
	0xf3c714
	0xfa3475
	0xfbe749
	0xf9c706
	0xf6da30
	0xf6ed54
	0x1395744
	0x139091a
	0x114c322
	0x113c458
	0x11431dd
	0x112c408
	0x112c5cc
	0x111675a
	0x75945d49
	0x7705d6db
	0x7705d661

4. Closing browser.


In [303]:
# Test cell 2, get_min_minute function needed for real analysis
from bs4 import BeautifulSoup
import re
print("\n--- Start Analysis")
soup = BeautifulSoup(html, 'html.parser')
events = soup.find_all('div', class_='event')
scorebox = soup.find('div', class_='scorebox')
card_classes = ['event_icon yellow_red_card', 'event_icon red_card']

events_goals = []
for team_event_container in events: 
    individual_events = team_event_container.find_all('div', recursive=False)    
    goal_events = []
    for item in individual_events:
        has_card = any(item.find('div', class_=card_class) for card_class in card_classes)
        if not has_card:
            goal_events.append(item)
            
    events_goals.append(goal_events)

def get_min_minute(event_div_list):
    text = ' '.join([div.get_text() for div in event_div_list])
    matches = re.findall(r'(\d+)(\+\d+)?[\’]', text)
    return min([int(m[0]) for m in matches])

    
minute_home = get_min_minute(events_goals[0])
minute_away = get_min_minute(events_goals[1])

print(f"first goal minute home: {minute_home}")
print(f"first goal minute away: {minute_away}")



--- Start Analysis


IndexError: list index out of range

In [ ]:
# Needed
import pandas as pd
# Read in df
df = pd.read_csv(r'Processed_full_stats/Full_df/full_df_8.csv')
# Filter games where Home_goals or Away_goals is 0 or both are 0
no_both_df = df[(df['Home_goals'] == 0) | (df['Away_goals'] == 0)]
print(f"number games with only 1 team or no teams scoring: {len(no_both_df)}")
# Filter rest games where Home_goals and Away_goals are both greater than 0
rest_df = df[(df['Home_goals'] > 0) & (df['Away_goals'] > 0)]
print(f"number games with both teams scoring: {len(rest_df)}")
# Add new column indicating if other team scored, called goal_scored_other. Fill with 0 for no_both_df and 1 for rest_df
no_both_df['goal_scored_other'] = 0
rest_df['goal_scored_other'] = 1
# Add new column with time of first goal other team, called time_first_goal_other. Fill with NaN for no_both_df. 
no_both_df['time_first_goal_other'] = pd.NA
# check newly made columns
print(no_both_df[['Home_goals', 'Away_goals', 'goal_scored_other', 'time_first_goal_other']].head())
print(rest_df[['Home_goals', 'Away_goals', 'goal_scored_other']].head())

# For rest_df, we will make a function to get the time based on the try-out cell above

number games with only 1 team or no teams scoring: 5467
number games with both teams scoring: 6674
    Home_goals  Away_goals  goal_scored_other time_first_goal_other
0            8           0                  0                  <NA>
6            3           0                  0                  <NA>
8            0           0                  0                  <NA>
11           1           0                  0                  <NA>
12           2           0                  0                  <NA>
   Home_goals  Away_goals  goal_scored_other
1           1           3                  1
2           1           1                  1
3           1           4                  1
4           2           3                  1
5           2           3                  1


C:\Users\kian3\AppData\Local\Temp\ipykernel_41096\3711165931.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_both_df['goal_scored_other'] = 0
C:\Users\kian3\AppData\Local\Temp\ipykernel_41096\3711165931.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rest_df['goal_scored_other'] = 1
C:\Users\kian3\AppData\Local\Temp\ipykernel_41096\3711165931.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

In [ ]:
# needed
print("start browser")
options = uc.ChromeOptions() 
driver = uc.Chrome(options=options)
card_classes = ['event_icon yellow_red_card', 'event_icon red_card']

def get_min_minute(event_div_list):
    text = ' '.join([div.get_text() for div in event_div_list])
    matches = re.findall(r'(\d+)(\+\d+)?[\’]', text)
    return min([int(m[0]) for m in matches])

def first_goal_minute_other(row, driver):
    url = row['link']
    driver.get(url)
    time.sleep(random.uniform(6, 8))
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    scorebox = soup.find('div', class_='scorebox')
    events = scorebox.find_all('div', class_='event')
    events_goals = []
    individual_events_home = events[0].find_all('div', recursive=False)   
    goal_events_home = []
    for event in individual_events_home:
        has_card = any(event.find('div', class_=card_class) for card_class in card_classes)
        if not has_card:
            goal_events_home.append(event)        
    events_goals.append(goal_events_home)
    individual_events_away = events[1].find_all('div', recursive=False)   
    goal_events_away = []
    for event in individual_events_away:
        has_card = any(event.find('div', class_=card_class) for card_class in card_classes)
        if not has_card:
            goal_events_away.append(event)  
    events_goals.append(goal_events_away)
    minute_home = get_min_minute(events_goals[0])
    minute_away = get_min_minute(events_goals[1])
    if minute_home < minute_away:
        row['time_first_goal_other'] = minute_away
    else:
        row['time_first_goal_other'] = minute_home
    return row


test_df = rest_df.head(2).copy()
test_df = test_df.apply(first_goal_minute_other, axis=1, args=(driver,))

print(test_df[['link', 'Home_goals', 'Away_goals', 'first_goal_minute', 'goal_scored_other', 'time_first_goal_other']])


start browser
                                                link  Home_goals  Away_goals  \
1  https://fbref.com/en/matches/74ee6b62/Union-Be...           1           3   
2  https://fbref.com/en/matches/e10b719d/Eintrach...           1           1   

   first_goal_minute  goal_scored_other  time_first_goal_other  
1                 40                  1                     75  
2                 51                  1                     62  


In [311]:
# Cell to scrape data for rest_df
# We will aply it to the whole rest_df in small parts. We will do per 30 games and concatenate all parts at the end (robust in case of faillure somewhere).
parts = []
part_size = 20
print("start browser")
options = uc.ChromeOptions() 
driver = uc.Chrome(options=options)

card_classes = ['event_icon yellow_red_card', 'event_icon red_card']

def get_min_minute(event_div_list):
    text = ' '.join([div.get_text() for div in event_div_list])
    matches = re.findall(r'(\d+)(\+\d+)?[\’]', text)
    return min([int(m[0]) for m in matches])

def first_goal_minute_other(row, driver):
    url = row['link']
    driver.get(url)
    time.sleep(random.uniform(6, 8))
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    scorebox = soup.find('div', class_='scorebox')
    events = scorebox.find_all('div', class_='event')
    events_goals = []
    individual_events_home = events[0].find_all('div', recursive=False)   
    goal_events_home = []
    for event in individual_events_home:
        has_card = any(event.find('div', class_=card_class) for card_class in card_classes)
        if not has_card:
            goal_events_home.append(event)        
    events_goals.append(goal_events_home)
    individual_events_away = events[1].find_all('div', recursive=False)   
    goal_events_away = []
    for event in individual_events_away:
        has_card = any(event.find('div', class_=card_class) for card_class in card_classes)
        if not has_card:
            goal_events_away.append(event)  
    events_goals.append(goal_events_away)
    minute_home = get_min_minute(events_goals[0])
    minute_away = get_min_minute(events_goals[1])
    if minute_home < minute_away:
        row['time_first_goal_other'] = minute_away
    else:
        row['time_first_goal_other'] = minute_home
    return row


for start in range(0, 285, part_size):
    end = start + part_size
    part_df = rest_df.iloc[start:end].copy()
    part_df = part_df.apply(first_goal_minute_other, axis=1, args=(driver,))
    parts.append(part_df)
final_rest_df27 = pd.concat(parts)

start browser


In [ ]:
# First done in small parts, changed names constantly to concatenate later
#final_rest_df_first2 = final_rest_df.copy()
#final_rest_df_3_to_6 = final_rest_df2.copy()
#final_rest_df_7_to_14 = final_rest_df4.copy()
# final_rest_df_15_to_24 = final_rest_df3.copy()
# final_rest_df_25_to_44 = final_rest_df5.copy()
# final_rest_df_45_to_89 = final_rest_df6.copy()
# final_rest_df_90_to_189 = final_rest_df7.copy()
# final_rest_df_190_to_339 = final_rest_df8.copy()
# final_rest_df_340_to_459 = final_rest_df9.copy()
# final_rest_df_460_to_559 = final_rest_df10.copy()
# final_rest_df_560_to_699 = final_rest_df11.copy()
# final_rest_df_700_to_859 = final_rest_df12.copy()
# final_rest_df_860_to_1039 = final_rest_df13.copy()
# final_rest_df_1039_to_1239 = final_rest_df14.copy()
# final_rest_df_1239_to_1459 = final_rest_df15.copy()
# final_rest_df_1459_to_1699 = final_rest_df16.copy()
# final_rest_df_1700_to_1959 = final_rest_df17.copy()
# final_rest_df_1960_to_2239 = final_rest_df18.copy()
# final_rest_df_2240_to_2539 = final_rest_df19.copy()
# final_rest_df_2540_to_2739 = final_rest_df20.copy()
# final_rest_df_2740_to_3039 = final_rest_df21.copy()
# final_rest_df_3040_to_3319 = final_rest_df22.copy()
# final_rest_df_3320_to_3619 = final_rest_df23.copy()
# final_rest_df_3620_to_3919 = final_rest_df24.copy()
# final_rest_df_3920_to_4219 = final_rest_df25.copy()
# final_rest_df_4220_to_4299 = final_rest_df26.copy()

In [ ]:
# Sanity checks
#print(parts)
pd.concat([final_rest_df_first2, final_rest_df_3_to_6, final_rest_df_7_to_14, final_rest_df_15_to_24, final_rest_df_25_to_44, final_rest_df_45_to_89, final_rest_df_90_to_189, final_rest_df_190_to_339, final_rest_df_340_to_459,final_rest_df_460_to_559,
            final_rest_df_560_to_699, final_rest_df_700_to_859, final_rest_df_860_to_1039, final_rest_df_1039_to_1239, final_rest_df_1239_to_1459, final_rest_df_1459_to_1699, final_rest_df_1700_to_1959, final_rest_df_1960_to_2239, final_rest_df_2240_to_2539, 
            final_rest_df_2540_to_2739, final_rest_df_2740_to_3039, final_rest_df_3040_to_3319, final_rest_df_3320_to_3619, final_rest_df_3620_to_3919, final_rest_df_3920_to_4219, final_rest_df_4220_to_4299]).index.equals(rest_df.iloc[0:4599].index)
# print(final_rest_df6[['link', 'Home_goals', 'Away_goals', 'first_goal_minute', 'goal_scored_other', 'time_first_goal_other']])
# print(pd.concat([final_rest_df_first2, final_rest_df_3_to_6, final_rest_df_7_to_14, final_rest_df_15_to_24, final_rest_df_25_to_44, final_rest_df_45_to_89, final_rest_df_90_to_189, final_rest_df_190_to_339])[['link', 'Home_goals', 'Away_goals', 'first_goal_minute', 'goal_scored_other', 'time_first_goal_other']])

NameError: name 'final_rest_df_4300_to_4599' is not defined

In [ ]:
# Save first 189 rows for now
pd.concat([final_rest_df_first2, final_rest_df_3_to_6, final_rest_df_7_to_14, final_rest_df_15_to_24, final_rest_df_25_to_44, final_rest_df_45_to_89, final_rest_df_90_to_189, final_rest_df_190_to_339, final_rest_df_340_to_459, final_rest_df_460_to_559, final_rest_df_560_to_699, final_rest_df_700_to_859, final_rest_df_860_to_1039, final_rest_df_1039_to_1239, final_rest_df_1239_to_1459, final_rest_df_1459_to_1699, final_rest_df_1700_to_1959, final_rest_df_1960_to_2239, final_rest_df_2240_to_2539, final_rest_df_2540_to_2739, final_rest_df_2740_to_3039, final_rest_df_3040_to_3319, final_rest_df_3320_to_3619, final_rest_df_3620_to_3919, final_rest_df_3920_to_4219])\
  .to_csv(r"Processed_full_stats\Full_df\Full_df_other_first_4219.csv", index=False)

In [ ]:
df_tim = pd.read_csv(r"Processed_full_stats\Full_df\final_3918tofinal.csv")
print(df_tim)
print(rest_df)

      Unnamed: 0.1  Unnamed: 0  \
0                0        6911   
1                1        6918   
2                2        6919   
3                3        6920   
4                4        6921   
...            ...         ...   
2570           295       12129   
2571           296       12132   
2572           297       12133   
2573           298       12136   
2574           299       12139   

                                            comp_season       Round    Wk  \
0         Processed_full_stats/Serie A/Serie A_2122.csv     Serie A  31.0   
1         Processed_full_stats/Serie A/Serie A_2122.csv     Serie A  32.0   
2         Processed_full_stats/Serie A/Serie A_2122.csv     Serie A  32.0   
3         Processed_full_stats/Serie A/Serie A_2122.csv     Serie A  32.0   
4         Processed_full_stats/Serie A/Serie A_2122.csv     Serie A  32.0   
...                                                 ...         ...   ...   
2570  Processed_full_stats/Eredivisie/Eredivisie_202

In [315]:
#print(df_tim.head())
#print(rest_df.iloc[3918:])
#df_3919 = pd.read_csv(r"Processed_full_stats\Full_df\Full_df_other_first_3919.csv")
# concatenate df_tim and df_3919, delete duplicates based on link column
# df_merged = pd.concat([df_tim, df_3919]).drop_duplicates(subset=['link'])
# Check if all rows from df_merged are in rest_df
# print(df_merged['link'].isin(rest_df['link']).all())
# Take out games from rest_df that are not in df_merged
#rows_left = rest_df['link'].isin(df_merged['link'])
#rows_left = rest_df[~rows_left]

# We will call rows_left as rest_df and apply the function from above
#rest_df = rows_left.copy()
# Concatenate final_rest_df27 to merged df
#final_df = pd.concat([df_merged, final_rest_df27])
#print(final_df)
# concatenate final_df and no_both_df
#full_final_df = pd.concat([final_df, no_both_df])
#print(full_final_df)
full_final_df.to_csv(r"Processed_full_stats\Full_df\full_df_with_other_first_goal_time.csv", index=False)